# Opion — Discard Training (T4 GPU)

The bot also needs to learn **which of its 3 hole cards to throw away** mid-hand.
We recorded 4.4M teacher discards; this trains a small 3-way head on them.

Same pipeline as the action model: `packed.tar` → `packed/{w1,w2}/sb*/disc.npz`.
Targets are **hard labels** (the card the teacher actually discarded — teacher
discards are deterministic heuristics, no soft labels exist for them).

## Steps
1. Runtime → **T4 GPU**
2. Run cells top-to-bottom (data pulls from your Drive via gdown, same file as before)
3. Smoke run (cell 8, ~1 min) → full training (cell 9, ~15-20 min)
4. Cell 11 downloads `final_disc.pt` — send it back with `final_act.pt`

In [ ]:
# 1. GPU check
import torch
assert torch.cuda.is_available(), "NO GPU! Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0), "| VRAM:", round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), "GB")

In [ ]:
# 2. Keepalive: free -h every 5 minutes (background thread)
import threading, time, subprocess
def _keepalive():
    while True:
        print(subprocess.run(["free", "-h"], capture_output=True, text=True).stdout, flush=True)
        time.sleep(300)
threading.Thread(target=_keepalive, daemon=True).start()
print("keepalive armed")

In [ ]:
# 3. Get the dataset (Drive, same packed.tar as before)
!gdown 1AgcO3IHlhFc4czzr1fU_EG39Bh7809TA -O packed.tar

In [ ]:
# 4. Extract (skips if already there)
import tarfile, glob, os

if not glob.glob("packed/*/*/disc.npz"):
    with tarfile.open("packed.tar") as tf:
        tf.extractall("/content")

shards = sorted(glob.glob("packed/*/*/disc.npz"))
print(len(shards), "shards found")   # should print: 40 shards found

In [ ]:
# 5. Discard model: same trunk style, 8 card tokens + 16 ctx dims, 3-way head
import torch
import torch.nn as nn

N_TOK_D, N_CTX_D, VOCAB = 8, 16, 85

class Block(nn.Module):
    def __init__(self, d, h, f):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.h = h
        self.ln2 = nn.LayerNorm(d)
        self.fc1 = nn.Linear(d, f)
        self.fc2 = nn.Linear(f, d)
        self.act = nn.GELU()
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).chunk(3, dim=-1)
        q = q.view(B, T, self.h, -1).transpose(1, 2)
        k = k.view(B, T, self.h, -1).transpose(1, 2)
        v = v.view(B, T, self.h, -1).transpose(1, 2)
        att = torch.softmax(q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5), -1)
        x = x + self.proj((att @ v).transpose(1, 2).reshape(B, T, D))
        x = x + self.fc2(self.act(self.fc1(self.ln2(x))))
        return x

class DiscPolicy(nn.Module):
    """tokens: [B,8] card ids (0=pad), ctx: [B,16] -> logits [B,3]
    (which of the 3 hole cards to discard)."""
    def __init__(self, d=128, layers=4, heads=4, f=512):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, d)
        self.pos = nn.Embedding(N_TOK_D, d)
        self.ctx_proj = nn.Sequential(nn.LayerNorm(N_CTX_D), nn.Linear(N_CTX_D, d))
        self.blocks = nn.ModuleList(Block(d, heads, f) for _ in range(layers))
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, 3)
    def forward(self, tokens, ctx):
        x = self.emb(tokens.long()) + self.pos.weight[None, :N_TOK_D]
        x = x + self.ctx_proj(ctx.float())[:, None, :]
        for b in self.blocks:
            x = b(x)
        return self.head(self.ln_f(x).mean(dim=1))
    def n_params(self):
        return sum(p.numel() for p in self.parameters())

m = DiscPolicy()
print("params:", f"{m.n_params():,}")

In [ ]:
# 6. Data: disc.npz shards -> GPU tensors
import numpy as np

def load_disc(parts):
    files = sorted(glob.glob("packed/*/*/disc.npz"))
    cols, n = {}, 0
    for f in files:
        z = np.load(f, allow_pickle=False)
        sel = np.isin(z["part"], parts)
        for k in ("tokens", "ctx", "target", "street"):
            cols.setdefault(k, []).append(z[k][sel])
        n += int(sel.sum())
    return {k: np.concatenate(v) for k, v in cols.items()}, n

def to_gpu(d):
    return {k: torch.from_numpy(v.astype(np.int64 if k in ("tokens", "target", "street") else np.float32)).cuda()
            for k, v in d.items()}

tr_raw, n_tr = load_disc((0,))
va_raw, n_va = load_disc((1,))
va_raw = {k: v[:100_000] for k, v in va_raw.items()}
tr, va = to_gpu(tr_raw), to_gpu(va_raw)
print(f"train={n_tr:,}  val={min(n_va, 100_000):,}")
tg = np.bincount(tr["target"].cpu().numpy(), minlength=3)
st = np.bincount(tr["street"].cpu().numpy(), minlength=4)
print("target distribution (which card discarded):", (tg / tg.sum()).round(3).tolist())
print("streets:", (st / st.sum()).round(3).tolist(), "(flop/turn/river + derived)")

In [ ]:
# 7. Trainer (hard-label cross-entropy)
import time, json, os

def evaluate(model, va, batch=16384):
    model.eval()
    acc = n = 0
    per_street = np.zeros(4)
    with torch.no_grad():
        for i in range(0, len(va["target"]), batch):
            sl = slice(i, i + batch)
            logits = model(va["tokens"][sl], va["ctx"][sl]).float()
            pred = logits.argmax(-1)
            hit = (pred == va["target"][sl])
            acc += int(hit.sum()); n += len(hit)
            s = va["street"][sl]
            for st_ in range(4):
                m = s == st_
                cnt = int(m.sum())
                if cnt:
                    per_street[st_] += hit[m].float().sum().item() / cnt * cnt
    model.train()
    # per-street accuracy: hits per street / records per street
    tot = np.zeros(4)
    for i in range(0, len(va["target"]), 1 << 20):
        sl = slice(i, i + (1 << 20))
        for st_ in range(4):
            tot[st_] += int((va["street"][sl] == st_).sum())
    return acc / n, per_street / np.maximum(tot, 1), tot

def run_training(epochs=3.0, batch=4096, lr=3e-4, max_steps=0, out="/content/ckpt_disc"):
    os.makedirs(out, exist_ok=True)
    model = DiscPolicy().cuda()
    print("params:", f"{model.n_params():,}")
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = int((n_tr // batch) * epochs)
    if max_steps: total = min(total, max_steps)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=total, pct_start=0.05)
    print(f"steps={total}  batch={batch}")
    t0 = time.time()
    for step in range(1, total + 1):
        idx = torch.randint(0, n_tr, (batch,), device="cuda")
        logits = model(tr["tokens"][idx], tr["ctx"][idx]).float()
        loss = torch.nn.functional.cross_entropy(logits, tr["target"][idx])
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step(); sched.step()
        if step % 100 == 0 or step == total:
            acc = (logits.argmax(-1) == tr["target"][idx]).float().mean()
            print(f"step {step}/{total}  loss={loss.item():.4f}  acc={acc.item():.3f}  "
                  f"lr={sched.get_last_lr()[0]:.1e}  {time.time()-t0:.0f}s", flush=True)
        if step % 500 == 0 or step == total:
            a, _, ps = evaluate(model, va)
            print(f"  VAL step {step}: acc={a:.3f}  per-street={np.round(ps, 3).tolist()}", flush=True)
            torch.save({"model": model.state_dict(), "step": step}, f"{out}/last.pt")
    a, _, ps = evaluate(model, va)
    final = dict(acc=a, per_street=ps.tolist(), steps=total, wall_min=round((time.time()-t0)/60, 1))
    torch.save({"model": model.state_dict(), "final": final}, f"{out}/final_disc.pt")
    print("FINAL:", json.dumps(final))
    return model, final

print("trainer ready")

In [ ]:
# 8. SMOKE RUN (~200 steps, ~1 min)
model_smoke, fs = run_training(epochs=1.0, batch=512, max_steps=200, out="/content/ckpt_disc_smoke")
print("SMOKE_OK" if fs["acc"] > 0.4 else "SMOKE_SUSPICIOUS")

In [ ]:
# 9. FULL TRAINING (~15-20 min)
model_disc, final_disc = run_training(epochs=3.0, batch=4096, lr=3e-4, out="/content/ckpt_disc")

## How to read the result

- **acc** = how often the student throws the exact same card the teacher throws.
- Teacher discards are a **deterministic heuristic** (weakest card by hand-strength),
  so a good student should reach **very high accuracy (90%+)** — much easier than the
  6-way betting policy.
- per-street = accuracy on flop / turn / river separately (street index 1/2/3).
- If acc stalls below ~80%, something's off — send me the output.
- NOTE: a student that just copies the discard heuristic exactly is *fine* — the
  discard is not where poker edge lives; the betting policy (already trained) is.

In [ ]:
# 10. Download the discard model
from google.colab import files
files.download("/content/ckpt_disc/final_disc.pt")